Giai đoạn 4.2: Collaborative Filtering

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors

In [2]:
movies = pd.read_csv('../data/processed/movies_features.csv')

with open('../models_artifacts/user_item_matrix.pkl', 'rb') as f:
    user_item_matrix = pickle.load(f)

print("user_item_matrix:", user_item_matrix.shape)
user_item_matrix.head(3)

user_item_matrix: (671, 3493)


tmdbId,5,11,12,13,14,15,16,18,19,21,...,273248,273481,281957,286217,293660,312221,314040,314365,318846,333371
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,3.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Bước 4.2.1: Xây mô hình tìm user tương tự

In [3]:
# Dùng cosine similarity giữa các user (mỗi user là 1 vector rating trên các phim)
user_knn = NearestNeighbors(metric='cosine', algorithm='brute')
user_knn.fit(user_item_matrix.values)

print("Đã fit KNN trên", user_item_matrix.shape[0], "user")

Đã fit KNN trên 671 user


Bước 4.2.2. Hàm gợi ý Collaborative Filtering

In [4]:
def get_cf_recommendations(user_id, top_n=5, k_neighbors=10):
    if user_id not in user_item_matrix.index:
        print(f"User {user_id} không có trong ma trận (cold-start)")
        return pd.DataFrame()
    
    user_pos = user_item_matrix.index.get_loc(user_id)
    user_vector = user_item_matrix.iloc[user_pos].values.reshape(1, -1)
    
    # Tìm k user gần nhất (bỏ qua chính user_id ở vị trí đầu)
    distances, indices_knn = user_knn.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
    similar_user_positions = indices_knn.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]  # cosine similarity = 1 - cosine distance
    
    similar_users = user_item_matrix.iloc[similar_user_positions]
    
    # Tính điểm trung bình có trọng số (theo similarity) cho từng phim
    weighted_scores = similar_users.T.dot(similarities) / (similarities.sum() + 1e-9)
    
    # Loại các phim mà user_id đã xem/rate rồi
    already_rated = user_item_matrix.loc[user_id]
    candidate_scores = weighted_scores[already_rated == 0]
    
    top_movie_ids = candidate_scores.sort_values(ascending=False).head(top_n)
    
    result = movies[movies['id'].isin(top_movie_ids.index)][
        ['id', 'title', 'weighted_rating', 'vote_average']].copy()
    result['cf_score'] = result['id'].map(top_movie_ids)
    result = result.sort_values('cf_score', ascending=False).reset_index(drop=True)
    return result

get_cf_recommendations(1, top_n=5)

,id,title,weighted_rating,vote_average,cf_score
0,90,Beverly Hills Cop,6.760576,6.8,1.881282
1,240,The Godfather: Part II,8.273588,8.3,1.589783
2,238,The Godfather,8.483828,8.5,1.589783
3,2039,Moonstruck,6.522355,6.7,1.269406
4,278,The Shawshank Redemption,8.488325,8.5,1.267557


Bước 4.2.3: Kiểm định với vài user khác nhau

In [5]:
for uid in [1, 5, 50, 300]:
    print(f"--- Người giống bạn đang xem (user_id={uid}) ---")
    res = get_cf_recommendations(uid, top_n=5)
    if not res.empty:
        print(res[['title', 'cf_score']].to_string(index=False))
    print()

--- Người giống bạn đang xem (user_id=1) ---
                   title  cf_score
       Beverly Hills Cop  1.881282
  The Godfather: Part II  1.589783
           The Godfather  1.589783
              Moonstruck  1.269406
The Shawshank Redemption  1.267557

--- Người giống bạn đang xem (user_id=5) ---
                                            title  cf_score
                                  The Incredibles  4.142426
The Lord of the Rings: The Fellowship of the Ring  3.971176
                         The Shawshank Redemption  3.891413
                               The Princess Bride  3.862592
    The Lord of the Rings: The Return of the King  3.761705

--- Người giống bạn đang xem (user_id=50) ---
                   title  cf_score
           The Lion King  3.650407
          Mrs. Doubtfire  3.222812
              The Client  2.631135
  Star Trek: Generations  2.599850
The Shawshank Redemption  2.471903

--- Người giống bạn đang xem (user_id=300) ---
                                  

Bước 4.2.4: Kiểm tra cold-start (user không tồn tại)

In [6]:
get_cf_recommendations(999999, top_n=5)

User 999999 không có trong ma trận (cold-start)


""


Bước 4.2.5: Đo thời gian chạy

In [7]:
import time
start = time.time()
_ = get_cf_recommendations(1, top_n=5)
print(f"Thời gian: {time.time() - start:.3f} giây")

Thời gian: 0.052 giây


In [8]:
import pickle

with open('../models_artifacts/user_knn_model.pkl', 'wb') as f:
    pickle.dump(user_knn, f)

print("Đã lưu user_knn_model.pkl")

Đã lưu user_knn_model.pkl
